In [0]:
# Phase 4: Gold Business Marts
# Purpose: Create analytics-ready aggregation tables for BI/stakeholder consumption
# Estimated time: ~1 hour

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType
from pyspark.sql.functions import (
    col, F, sum as F_sum, avg as F_avg, count as F_count, min as F_min, max as F_max,
    date_format, datediff, concat_ws, round as F_round, desc, asc, coalesce, lit,
    year, month, date_format, when, rank, row_number
)
from pyspark.sql import Window

print("✓ Imports loaded")
print("✓ Ready to build Gold layer business marts")

In [0]:
def save_gold_table(df, table_name, optimize=True):
    """Save a Gold table with optional ZORDER optimization."""
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)
    
    row_count = spark.table(table_name).count()
    print(f"✓ {table_name}: {row_count} rows saved")
    
    return row_count

In [0]:
# Gold Mart 1: Monthly Revenue Trend
# Purpose: Track GMV (Gross Merchandise Value) and order volume by month
# Used by: Finance, Executive dashboards

revenue_by_month = (
    spark.table("workspace.olist_silver.orders")
    .join(
        spark.table("workspace.olist_silver.order_items"),
        on="order_id",
        how="inner"
    )
    .withColumn(
        "year_month",
        date_format(col("order_purchase_timestamp"), "yyyy-MM")
    )
    .groupBy("year_month")
    .agg(
        F_count("distinct order_id").alias("order_count"),
        F_sum("total_item_value").cast(DecimalType(15, 2)).alias("total_revenue_brl"),
        F_avg("delivery_days").cast(DecimalType(5, 2)).alias("avg_delivery_days"),
        F_max("total_item_value").cast(DecimalType(15, 2)).alias("max_order_value_brl")
    )
    .orderBy("year_month")
)

save_gold_table(revenue_by_month, "workspace.olist_gold.revenue_by_month")
display(revenue_by_month)

In [0]:
# Gold Mart 2: Category Performance
# Purpose: Rank product categories by revenue, order count, and avg review score
# Used by: Product management, Merchandising

top_categories = (
    spark.table("workspace.olist_silver.order_items")
    .join(
        spark.table("workspace.olist_silver.products"),
        on="product_id",
        how="inner"
    )
    .join(
        spark.table("workspace.olist_silver.orders").select("order_id"),
        on="order_id",
        how="inner"
    )
    .groupBy(
        col("product_category_name_english").alias("category")
    )
    .agg(
        F_count("distinct order_id").alias("order_count"),
        F_sum("total_item_value").cast(DecimalType(15, 2)).alias("total_revenue_brl"),
        F_avg("total_item_value").cast(DecimalType(10, 2)).alias("avg_item_value_brl"),
        F_count("product_id").alias("items_sold"),
        F_count("distinct product_id").alias("unique_products")
    )
    .withColumn(
        "revenue_rank",
        rank().over(Window.orderBy(desc("total_revenue_brl")))
    )
    .orderBy("revenue_rank")
)

save_gold_table(top_categories, "workspace.olist_gold.top_categories")
display(top_categories.limit(20))

In [0]:
# Gold Mart 3: Geographic Performance
# Purpose: Revenue, orders, and customer metrics by Brazilian state
# Used by: Regional managers, Geographic expansion strategy

state_performance = (
    spark.table("workspace.olist_silver.orders")
    .join(
        spark.table("workspace.olist_silver.customers"),
        on="customer_id",
        how="inner"
    )
    .join(
        spark.table("workspace.olist_silver.order_items"),
        on="order_id",
        how="inner"
    )
    .groupBy(
        col("customer_state").alias("state")
    )
    .agg(
        F_count("distinct order_id").alias("order_count"),
        F_sum("total_item_value").cast(DecimalType(15, 2)).alias("total_revenue_brl"),
        F_count("distinct customer_id").alias("unique_customers"),
        F_avg("total_item_value").cast(DecimalType(10, 2)).alias("avg_order_value_brl"),
        F_avg("delivery_delay_days").cast(DecimalType(5, 2)).alias("avg_delivery_delay_days")
    )
    .withColumn(
        "revenue_rank",
        rank().over(Window.orderBy(desc("total_revenue_brl")))
    )
    .orderBy("revenue_rank")
)

save_gold_table(state_performance, "workspace.olist_gold.state_performance")
display(state_performance)

In [0]:
# Gold Mart 4: Payment Method Analysis
# Purpose: Understanding payment behavior across the customer base
# Used by: Finance, Risk management, Customer experience

payment_method_mix = (
    spark.table("workspace.olist_silver.payments")
    .groupBy("primary_payment_type")
    .agg(
        F_count("order_id").alias("order_count"),
        F_sum("total_payment_value").cast(DecimalType(15, 2)).alias("total_value_brl"),
        F_avg("total_payment_value").cast(DecimalType(10, 2)).alias("avg_order_value_brl"),
        F_avg("max_installments").cast(DecimalType(5, 2)).alias("avg_installments"),
        F_count(when(col("is_split_payment") == 1, 1)).alias("split_payment_count")
    )
    .withColumn(
        "order_percentage",
        F_round((col("order_count") / F_sum("order_count").over(Window.partitionBy())) * 100, 2)
    )
    .withColumn(
        "revenue_percentage",
        F_round((col("total_value_brl") / F_sum("total_value_brl").over(Window.partitionBy())) * 100, 2)
    )
    .orderBy(desc("order_count"))
)

save_gold_table(payment_method_mix, "workspace.olist_gold.payment_method_mix")
display(payment_method_mix)

In [0]:
# Gold Mart 5: Delivery Performance
# Purpose: Logistics health metrics — on-time delivery rate, delays, geographic variance
# Used by: Operations, Logistics partners

delivery_performance = (
    spark.table("workspace.olist_silver.orders")
    .join(
        spark.table("workspace.olist_silver.customers"),
        on="customer_id",
        how="inner"
    )
    .groupBy(
        col("customer_state").alias("state")
    )
    .agg(
        F_count("order_id").alias("order_count"),
        F_avg("delivery_days").cast(DecimalType(5, 2)).alias("avg_delivery_days"),
        F_avg("delivery_delay_days").cast(DecimalType(5, 2)).alias("avg_delay_days"),
        F_count(when(col("delivery_delay_days") <= 0, 1)).alias("early_or_ontime_count"),
        F_count(when(col("delivery_delay_days") > 0, 1)).alias("late_count")
    )
    .withColumn(
        "ontime_rate_pct",
        F_round((col("early_or_ontime_count") / col("order_count")) * 100, 2)
    )
    .orderBy(desc("order_count"))
)

save_gold_table(delivery_performance, "workspace.olist_gold.delivery_performance")
display(delivery_performance)

In [0]:
# Gold Mart 6: Customer Satisfaction by Category
# Purpose: Track review scores across product categories to identify quality issues
# Used by: Product quality, Customer success

review_sentiment_by_category = (
    spark.table("workspace.olist_silver.reviews")
    .join(
        spark.table("workspace.olist_silver.orders").select("order_id"),
        on="order_id",
        how="inner"
    )
    .join(
        spark.table("workspace.olist_silver.order_items").select("order_id", "product_id"),
        on="order_id",
        how="inner"
    )
    .join(
        spark.table("workspace.olist_silver.products").select("product_id", "product_category_name_english"),
        on="product_id",
        how="inner"
    )
    .groupBy(
        col("product_category_name_english").alias("category")
    )
    .agg(
        F_count("order_id").alias("review_count"),
        F_avg("review_score").cast(DecimalType(3, 2)).alias("avg_score"),
        F_min("review_score").alias("min_score"),
        F_max("review_score").alias("max_score"),
        F_count(when(col("review_score") <= 2, 1)).alias("negative_reviews"),
        F_count(when(col("review_score") >= 4, 1)).alias("positive_reviews"),
        F_count(when(col("has_comment") == 1, 1)).alias("commented_review_count")
    )
    .withColumn(
        "sentiment_rank",
        rank().over(Window.orderBy(asc("avg_score")))
    )
    .orderBy("sentiment_rank")
)

save_gold_table(review_sentiment_by_category, "workspace.olist_gold.review_sentiment_by_category")
display(review_sentiment_by_category)

In [0]:
# Gold Mart 7: Seller Performance Leaderboard
# Purpose: Identify top-performing sellers; track their contribution to platform
# Used by: Seller management, Marketplace strategy

top_sellers = (
    spark.table("workspace.olist_silver.order_items")
    .join(
        spark.table("workspace.olist_silver.sellers"),
        on="seller_id",
        how="inner"
    )
    .groupBy(
        col("seller_id"),
        col("seller_city"),
        col("seller_state")
    )
    .agg(
        F_count("distinct order_id").alias("order_count"),
        F_count("*").alias("items_sold"),
        F_sum("total_item_value").cast(DecimalType(15, 2)).alias("total_revenue_brl"),
        F_avg("total_item_value").cast(DecimalType(10, 2)).alias("avg_item_value_brl")
    )
    .withColumn(
        "revenue_rank",
        rank().over(Window.orderBy(desc("total_revenue_brl")))
    )
    .filter(col("revenue_rank") <= 100)
    .orderBy("revenue_rank")
)

save_gold_table(top_sellers, "workspace.olist_gold.top_sellers")
display(top_sellers.limit(20))

In [0]:
# Validation: Spot-check all Gold marts and verify row counts

import datetime

print("="*80)
print("PHASE 4: GOLD BUSINESS MARTS — VALIDATION")
print(f"Timestamp: {datetime.datetime.now()}")
print("="*80)

gold_tables = [
    "workspace.olist_gold.revenue_by_month",
    "workspace.olist_gold.top_categories",
    "workspace.olist_gold.state_performance",
    "workspace.olist_gold.payment_method_mix",
    "workspace.olist_gold.delivery_performance",
    "workspace.olist_gold.review_sentiment_by_category",
    "workspace.olist_gold.top_sellers"
]

for table in gold_tables:
    count = spark.table(table).count()
    print(f"✓ {table.split('.')[-1]}: {count} rows")

print("\n" + "="*80)
print("All Gold marts built successfully!")
print("Ready for Phase 5: RFM Segmentation")
print("="*80)